# OpenAI Quick Reference

This notebook is a compact companion to the scripts in `01_openai/scripts`.
It focuses on the most common actions you will repeat when working with the OpenAI Python SDK:

- create a client
- generate plain text
- parse structured outputs
- stream text incrementally
- create embeddings
- send richer message inputs


In [1]:
import math
import sys
from pathlib import Path

from dotenv import load_dotenv
from pydantic import BaseModel, Field


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "01_openai" / "scripts").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current notebook path")


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / "01_openai" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))




## Setup

The helper module in `01_openai/scripts/openai_client.py` keeps these examples short and close to the rest of the labs.

In [2]:
from openai_client import (
    DEFAULT_TEXT_MODEL,
    generate_embedding,
    generate_structured,
    generate_text,
    generate_text_from_input,
    get_client,
    stream_response,
)

load_dotenv(PROJECT_ROOT / ".env")
client = get_client()
DEFAULT_TEXT_MODEL


'gpt-5-mini'

## 1. Plain Text Generation

Use `responses.create(...)` when you want normal text output.

In [9]:
answer = generate_text(
    "Explain what an API is in 3 short bullet points.",
    instructions="You are a concise Python tutor.",
    client=client,
)

print(answer)


- API = Application Programming Interface — a defined way for software components to communicate.  
- It exposes functions/endpoints and rules (inputs, outputs, authentication) so you can use features without knowing internal details.  
- Common for libraries, web services, and OS features; enables reuse, modularity, and integration.


## 2. Structured Outputs With Pydantic

Use `responses.parse(...)` when you want typed, validated output.

In [10]:
class LessonPlan(BaseModel):
    title: str
    difficulty: str
    estimated_minutes: int = Field(ge=1)
    key_points: list[str]


lesson_plan = generate_structured(
    "Create a short beginner lesson plan about Python lists.",
    output_model=LessonPlan,
    instructions="Return a concise educational plan.",
    client=client,
)

lesson_plan


LessonPlan(title='Introduction to Python Lists', difficulty='Beginner', estimated_minutes=30, key_points=['What a list is: ordered, mutable collection of items (heterogeneous allowed).', 'Creating lists: literals [] and using list()', 'Accessing elements: positive and negative indexing', 'Slicing lists to get sublists (start:stop:step)', 'Common methods: append(), insert(), pop(), remove(), extend()', 'Useful operations: len(), in, concatenation (+), repetition (*)', 'Sorting and reversing: sort(), sorted(), reverse()', 'Iteration over lists with for-loops and enumerate()', 'List comprehensions for concise transformations and filtering', 'Nested lists and cautions about mutability and aliasing (shallow vs deep copy)'])

## 3. Streaming

Streaming is useful when you want to display text progressively in a UI or CLI.

In [11]:
for delta in stream_response(
    "Write a short explanation of streaming responses in 3 sentences.",
    client=client,
):
    print(delta, end="")

print()


Streaming responses send parts of the output incrementally to the client as they are generated instead of waiting for the entire response to be ready. This reduces perceived latency and enables progressive rendering, real-time updates, or handling very large outputs without buffering the whole result. They’re implemented with techniques like HTTP chunked transfer, Server-Sent Events, or WebSockets and are commonly used for live feeds, long-running computations, and interactive chat or media streaming.


## 4. Embeddings

Embeddings turn text into vectors so you can compare semantic similarity.

In [12]:
def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b, strict=True))
    norm_a = math.sqrt(sum(value * value for value in vector_a))
    norm_b = math.sqrt(sum(value * value for value in vector_b))
    return dot_product / (norm_a * norm_b)


vector_a = generate_embedding("How do I sort a Python list?", client=client)
vector_b = generate_embedding(
    "What is the best way to order items in a Python list?",
    client=client,
)

cosine_similarity(vector_a, vector_b)


0.7462651782286209

## 5. Rich Inputs

When the input is more complex than one string, send a full Responses API payload.

In [13]:
messages = [
    {"role": "system", "content": "You are a concise coding assistant."},
    {"role": "user", "content": "Summarize what a Python tuple is in one paragraph."},
]

generate_text_from_input(messages, client=client)


'A Python tuple is an ordered, immutable collection type that can hold heterogeneous elements (numbers, strings, objects, even other tuples); because tuples are immutable, their elements cannot be changed after creation, which makes tuples hashable when all contained items are hashable and therefore usable as dictionary keys or set members. Tuples support indexing, slicing, iteration, concatenation and repetition, and are commonly created with parentheses (e.g., (1, 2)) or simply by separating values with commas (a single-element tuple requires a trailing comma: (1,)). They are generally lighter-weight and slightly faster than lists for fixed collections where modification is not needed.'

## Where To Go Next

Use the notebooks for quick iteration, then switch to the scripts when you want fuller end-to-end labs:

- `01_basic_openai_calls.py` for the basic request patterns
- `02_structured_output_scenarios.py` for typed and schema-based outputs
- `05_streaming_scenarios.py` for streaming event patterns
- `10_embeddings_scenarios.py` for similarity and semantic search
- `14_model_comparison_scenarios.py` for benchmarking trade-offs
